# Per-ticker volatility

The notebook fits a GARCH(1,1) to the adjusted-close log returns of each ticker with
enough history. It joins the 20-day dollar ADV and the company name to the fitted
parameters. Later sections compare GARCH with EWMA and with range-based estimators.

In [ ]:
import numpy as np
import plotly.express as px
import polars as pl
from arch import arch_model

from sdp import dal

wh = dal.warehouse()   # A read-only connection to the dbt warehouse.

In [ ]:
# dollar_volume is the raw close times the raw volume, so a split does not change it.
# The rows with a null adj_close_total stay, so that no return spans a gap.
prices = wh.sql("""
    select ticker, date, adj_close_total, dollar_volume
    from main_staging.stg_prices_adjusted
    order by ticker, date
""").pl()
prices.head()

In [ ]:
# 20-day dollar ADV: the 20 most recent sessions of each ticker.
adv = prices.sql("""
    with recent as (
        select ticker, dollar_volume, date,
               row_number() over (partition by ticker order by date desc) as rn
        from self
    )
    select ticker,
           avg(dollar_volume) as adv_20d_dollars,
           max(date)          as adv_as_of
    from recent
    where rn <= 20
    group by ticker
""")
adv.head()

In [ ]:
# One row for each ticker, with the name from its most recent partition. The join
# below needs exactly one row for each ticker.
names = dal.tickers().query("t", """
    select ticker, name
    from (select ticker, name,
                 row_number() over (partition by ticker order by date desc) as rn
          from t)
    where rn = 1
""").pl()
names.head()

## The parameters

GARCH(1,1) on returns scaled x100 (percent):

$$\sigma_t^2 = \omega + \alpha_1\,\varepsilon_{t-1}^2 + \beta_1\,\sigma_{t-1}^2$$

- **omega** ($\alpha_0$) - the constant.
- **alpha1** ($\alpha_1$) - reaction to the last shock.
- **beta1** ($\beta_1$) - carry-over of past variance.
- **persistence** $=\alpha_1+\beta_1$ - how slowly volatility mean-reverts. A value $\ge 1$ is non-stationary.
- **unconditional (equilibrium) variance** $=\omega/(1-\alpha_1-\beta_1)$ - the long-run level.
  The denominator uses **both** $\alpha_1$ and $\beta_1$, not $1-\beta_1$. Reported as
  annualised percent (`uncond_vol_ann_pct`).
- **sigma_now_ann** - the current conditional volatility (last day), annualised.
- **half_life** $=\ln(0.5)/\ln(\alpha_1+\beta_1)$ - sessions for a shock to decay halfway.

In [ ]:
MIN_OBS = 500   # The minimum return count for a fit. A higher value fits fewer.

# Log returns per ticker, then one percent-scale array per ticker.
rets = (prices
        .sort("ticker", "date")
        .with_columns(pl.col("adj_close_total").log().diff().over("ticker").alias("r"))
        .drop_nulls("r")
        .group_by("ticker", maintain_order=True)
        .agg(pl.col("r")))

items = [(t, np.asarray(r, dtype=float) * 100.0)
         for t, r in zip(rets["ticker"], rets["r"].to_list(), strict=True)
         if len(r) >= MIN_OBS]
print(f"{len(items)} tickers to fit (>= {MIN_OBS} return observations)")

In [ ]:
def fit_one(ticker: str, r: np.ndarray) -> dict:
    """Fit GARCH(1,1) on one percent-scale return series. A failed fit returns NaN
    parameters and does not raise."""
    out = {"ticker": ticker, "nobs": int(r.size), "omega": np.nan,
           "alpha1": np.nan, "beta1": np.nan, "persistence": np.nan,
           "uncond_vol_ann_pct": np.nan, "sigma_now_ann": np.nan,
           "half_life_days": np.nan, "loglik": np.nan, "aic": np.nan,
           "converged": False}
    try:
        res = arch_model(r, mean="Zero", vol="GARCH", p=1, q=1,
                         rescale=False).fit(disp="off", show_warning=False)
        p = res.params
        omega, a1, b1 = float(p["omega"]), float(p["alpha[1]"]), float(p["beta[1]"])
        persistence = a1 + b1
        lrv = omega / (1.0 - persistence) if persistence < 1.0 else np.nan
        out.update(
            omega=omega, alpha1=a1, beta1=b1, persistence=persistence,
            uncond_vol_ann_pct=(float(np.sqrt(lrv * 252.0))
                                if np.isfinite(lrv) and lrv > 0 else np.nan),
            sigma_now_ann=float(res.conditional_volatility[-1] * np.sqrt(252.0)),
            half_life_days=(float(np.log(0.5) / np.log(persistence))
                            if 0.0 < persistence < 1.0 else np.nan),
            loglik=float(res.loglikelihood), aic=float(res.aic),
            converged=(res.convergence_flag == 0),
        )
    except Exception:
        pass
    return out


try:
    from joblib import Parallel, delayed
    rows = Parallel(n_jobs=-1, backend="loky", batch_size=16)(
        delayed(fit_one)(t, r) for t, r in items)
except ImportError:
    rows = [fit_one(t, r) for t, r in items]   # serial fallback

garch = pl.DataFrame(rows)
garch.head()

In [ ]:
# One row per fitted ticker: parameters + 20-day dollar ADV + company name.
result = (garch
          .join(adv, on="ticker", how="left")
          .join(names, on="ticker", how="left")
          .select("ticker", "name", "adv_20d_dollars", "nobs",
                  "omega", "alpha1", "beta1", "persistence",
                  "uncond_vol_ann_pct", "sigma_now_ann", "half_life_days",
                  "loglik", "aic", "converged", "adv_as_of")
          .sort("adv_20d_dollars", descending=True, nulls_last=True))
result.head(30)

## Visualizations

### Single ticker

Change `TICKER` to plot a different name. These plots need the fitted volatility
path, so they fit that one ticker again.

In [ ]:
import plotly.graph_objects as go
from scipy import stats

TICKER = "SPY"          # change this for the single-ticker plots
ADV_FLOOR = 5_000_000    # dollars/day, for the cross-sectional plots below


def fit_full(ticker: str):
    """Fit GARCH(1,1) on one ticker. Each return keeps the date of its session."""
    r = (prices.filter(pl.col("ticker") == ticker).sort("date")
         .with_columns(ret=pl.col("adj_close_total").log().diff() * 100.0)
         .drop_nulls("ret"))
    ret = r["ret"].to_numpy()
    res = arch_model(ret, mean="Zero", vol="GARCH", p=1, q=1,
                     rescale=False).fit(disp="off", show_warning=False)
    return r["date"].to_numpy(), ret, res


dates, ret, res = fit_full(TICKER)
sigma = res.conditional_volatility          # daily, percent
p = res.params
persistence = p["alpha[1]"] + p["beta[1]"]
eq_var = p["omega"] / (1 - persistence)     # long-run variance, (percent)^2
eq_vol_ann = float(np.sqrt(eq_var * 252))   # percent per year

In [ ]:
# Returns inside the +/- 2 sigma_t band: the band widens in clusters the model tracks.
fig = go.Figure()
fig.add_trace(go.Scatter(x=dates, y=2 * sigma, mode="lines", name="+2 sigma_t",
                         line=dict(color="#1f6feb", width=1)))
fig.add_trace(go.Scatter(x=dates, y=-2 * sigma, mode="lines", name="-2 sigma_t",
                         line=dict(color="#1f6feb", width=1),
                         fill="tonexty", fillcolor="rgba(31,111,235,0.12)"))
fig.add_trace(go.Scatter(x=dates, y=ret, mode="lines", name="log return %",
                         line=dict(color="#444", width=0.7)))
fig.update_layout(title=f"{TICKER}: returns inside the +/- 2 sigma_t GARCH band",
                  yaxis_title="percent", height=420, legend_traceorder="reversed")
fig.show()

In [ ]:
# Annualised conditional vol, mean-reverting to the equilibrium level.
fig = go.Figure(go.Scatter(x=dates, y=sigma * np.sqrt(252), mode="lines",
                           name="sigma_t annualised", line=dict(color="#cf222e", width=1)))
fig.add_hline(y=eq_vol_ann, line_dash="dash", line_color="#1a7f37",
              annotation_text=f"equilibrium {eq_vol_ann:.1f}%/yr")
fig.update_layout(title=f"{TICKER}: annualised conditional volatility",
                  yaxis_title="percent per year", height=380)
fig.show()

In [ ]:
# QQ of raw returns and GARCH standardised residuals. Blue points near the line and
# grey points away from it show that GARCH explains the fat tails.
zr = (ret - ret.mean()) / ret.std()
qraw_x, qraw_y = stats.probplot(zr, dist="norm", fit=False)
qg_x, qg_y = stats.probplot(res.std_resid, dist="norm", fit=False)
lim = [min(qraw_x.min(), -4.0), max(qraw_x.max(), 4.0)]
fig = go.Figure()
fig.add_trace(go.Scatter(x=lim, y=lim, mode="lines", name="normal",
                         line=dict(color="#333", dash="dash")))
fig.add_trace(go.Scatter(x=qraw_x, y=qraw_y, mode="markers", name="raw returns",
                         marker=dict(size=3, color="#9aa0a6")))
fig.add_trace(go.Scatter(x=qg_x, y=qg_y, mode="markers", name="GARCH std resid",
                         marker=dict(size=3, color="#1f6feb")))
fig.update_layout(title=f"{TICKER}: QQ, raw returns vs GARCH standardised residuals",
                  xaxis_title="normal quantile", yaxis_title="sample quantile", height=460)
fig.show()

In [ ]:
# News impact curve: the next variance as a function of the last shock. Plain
# GARCH makes it symmetric. An asymmetric response in the data is a reason to fit
# GJR or EGARCH.
omega, a1, b1 = p["omega"], p["alpha[1]"], p["beta[1]"]
grid = np.linspace(-5 * np.sqrt(eq_var), 5 * np.sqrt(eq_var), 200)
nic = omega + a1 * grid**2 + b1 * eq_var     # sigma^2_{t-1} held at the long-run level
fig = go.Figure(go.Scatter(x=grid, y=nic, mode="lines", line=dict(color="#1a7f37")))
fig.update_layout(title=f"{TICKER}: news impact curve (symmetric for plain GARCH)",
                  xaxis_title="last shock eps_{t-1} (percent)",
                  yaxis_title="next sigma^2_t", height=380)
fig.show()

### Cross-sectional

Each plot uses only the fits with `converged == True` and an ADV of at least
`ADV_FLOOR`. This removes the unreliable fits of illiquid names.

In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess

cs = (result
      .filter(pl.col("converged")
              & (pl.col("adv_20d_dollars") >= ADV_FLOOR)
              & pl.col("persistence").is_finite())
      .to_pandas())
print(f"{len(cs)} converged fits above ${ADV_FLOOR:,.0f}/day ADV")


def adv_scatter(df, ycol, ytitle, yrange, title):
    """Scatter vs ADV on a log-x axis, with a lowess trend fit in log space."""
    d = df.dropna(subset=[ycol, "adv_20d_dollars"])
    lo = lowess(d[ycol].to_numpy(), np.log10(d["adv_20d_dollars"].to_numpy()), frac=0.3)
    fig = px.scatter(d, x="adv_20d_dollars", y=ycol, opacity=0.35,
                     hover_name="name", title=title)
    fig.add_trace(go.Scatter(x=10 ** lo[:, 0], y=lo[:, 1], mode="lines",
                             name="lowess", line=dict(color="#cf222e", width=2)))
    fig.update_xaxes(type="log", title="ADV ($/day, log)")
    fig.update_yaxes(title=ytitle, range=yrange)
    fig.update_layout(height=440)
    return fig

In [ ]:
# The (reaction, memory) map. Points near the dashed line are near-integrated.
fig = px.scatter(cs, x="alpha1", y="beta1", color="persistence",
                 color_continuous_scale="Viridis", opacity=0.5, hover_name="name",
                 hover_data=["ticker", "adv_20d_dollars", "half_life_days"],
                 title=f"alpha1 vs beta1 across {len(cs)} liquid names")
fig.add_trace(go.Scatter(x=[0, 1], y=[1, 0], mode="lines", name="alpha+beta = 1",
                         line=dict(color="#cf222e", dash="dash")))
fig.update_layout(height=520, xaxis_title="alpha1 (shock reaction)",
                  yaxis_title="beta1 (variance memory)")
fig.show()

In [ ]:
fig = px.histogram(cs, x="persistence", nbins=80,
                   title="Persistence (alpha1+beta1) across liquid names")
fig.add_vline(x=1.0, line_dash="dash", line_color="#cf222e",
              annotation_text="non-stationary")
fig.update_layout(height=380, yaxis_title="tickers")
fig.show()

In [ ]:
# Equilibrium volatility against size.
adv_scatter(cs, "uncond_vol_ann_pct", "equilibrium vol (%/yr)", [0, 150],
            "Equilibrium volatility vs 20-day ADV").show()

In [ ]:
# Volatility half-life against size.
adv_scatter(cs, "half_life_days", "half-life (sessions)", [0, 200],
            "Volatility half-life vs 20-day ADV").show()

In [ ]:
fig = px.histogram(cs.dropna(subset=["half_life_days"]), x="half_life_days",
                   nbins=80, range_x=[0, 200],
                   title="Volatility shock half-life across liquid names")
fig.update_layout(height=380, xaxis_title="half-life (sessions)", yaxis_title="tickers")
fig.show()

## EWMA and range-based realized volatility

EWMA is GARCH with omega = 0 and alpha + beta = 1. It gives a nowcast with no mean
reversion. The range estimators read the OHLC of one day and are more efficient
than a close-to-close estimator. Yang-Zhang also includes the overnight gap. All
values below are annualised percent over the latest `RV_WIN` sessions. They cover
every ticker, including the names that GARCH could not fit.

In [ ]:
LAMBDA = 0.94
ANN = np.sqrt(252) * 100.0

# EWMA(0.94) latest volatility for every ticker, vectorised.
ewma = (prices
        .sort("ticker", "date")
        .with_columns(r=pl.col("adj_close_total").log().diff().over("ticker"))
        .drop_nulls("r")
        .with_columns(ewvar=(pl.col("r") ** 2)
                      .ewm_mean(alpha=1 - LAMBDA, adjust=False).over("ticker"))
        .group_by("ticker")
        .agg(ewma_vol_ann=(pl.col("ewvar").last().sqrt() * ANN)))
ewma.head()

In [ ]:
RV_WIN = 20
LN2 = np.log(2.0)

ohlc = wh.sql("""
    select ticker, date,
           adj_open_split as o, adj_high_split as h,
           adj_low_split as l, adj_close_split as c
    from main_staging.stg_prices_adjusted
    where adj_open_split is not null and adj_high_split is not null
      and adj_low_split is not null and adj_close_split is not null
    order by ticker, date
""").pl()

# Per-day log ranges, then the per-day estimators. onr is the overnight return.
daily = (ohlc.with_columns(
            hl=(pl.col("h") / pl.col("l")).log(),
            co=(pl.col("c") / pl.col("o")).log(),
            ho=(pl.col("h") / pl.col("o")).log(),
            lo=(pl.col("l") / pl.col("o")).log(),
            hc=(pl.col("h") / pl.col("c")).log(),
            lc=(pl.col("l") / pl.col("c")).log(),
            onr=(pl.col("o") / pl.col("c").shift(1).over("ticker")).log())
         .with_columns(
            park=pl.col("hl") ** 2 / (4 * LN2),
            gk=0.5 * pl.col("hl") ** 2 - (2 * LN2 - 1) * pl.col("co") ** 2,
            rs=pl.col("ho") * pl.col("hc") + pl.col("lo") * pl.col("lc"))
         .with_columns(rn=pl.int_range(pl.len()).reverse().over("ticker")))

# Latest RV_WIN sessions per ticker. Yang-Zhang needs the overnight and
# open-close variances plus the average Rogers-Satchell.
k = 0.34 / (1.34 + (RV_WIN + 1) / (RV_WIN - 1))
rv = (daily.filter(pl.col("rn") < RV_WIN)
      .group_by("ticker")
      .agg(park_var=pl.col("park").mean(), gk_var=pl.col("gk").mean(),
           rs_var=pl.col("rs").mean(), ov=pl.col("onr").var(), oc=pl.col("co").var())
      .with_columns(yz_var=pl.col("ov") + k * pl.col("oc") + (1 - k) * pl.col("rs_var"))
      .select("ticker",
              parkinson_vol=(pl.col("park_var").clip(0).sqrt() * ANN),
              garman_klass_vol=(pl.col("gk_var").clip(0).sqrt() * ANN),
              rogers_satchell_vol=(pl.col("rs_var").clip(0).sqrt() * ANN),
              yang_zhang_vol=(pl.col("yz_var").clip(0).sqrt() * ANN)))
rv.head()

In [ ]:
# One row per ticker, every volatility side by side (annualised %): GARCH
# long-run and current, EWMA nowcast, and the four range estimators.
vol = (result.select("ticker", "name", "adv_20d_dollars", "converged",
                     "uncond_vol_ann_pct", "sigma_now_ann")
       .rename({"uncond_vol_ann_pct": "garch_equilibrium", "sigma_now_ann": "garch_now"})
       .join(ewma, on="ticker", how="left")
       .join(rv, on="ticker", how="left")
       .sort("adv_20d_dollars", descending=True, nulls_last=True))
vol.head(30)

In [ ]:
# Single ticker: EWMA reacts harder to the last shock, GARCH pulls back toward
# its equilibrium. Uses TICKER / ret / sigma / dates from the single-ticker cell.
ew_path = (pl.Series(ret ** 2).ewm_mean(alpha=1 - LAMBDA, adjust=False).sqrt().to_numpy()
           * np.sqrt(252))
fig = go.Figure()
fig.add_trace(go.Scatter(x=dates, y=sigma * np.sqrt(252), mode="lines",
                         name="GARCH sigma_t", line=dict(color="#cf222e", width=1)))
fig.add_trace(go.Scatter(x=dates, y=ew_path, mode="lines",
                         name=f"EWMA (lambda={LAMBDA})", line=dict(color="#1f6feb", width=1)))
fig.add_hline(y=eq_vol_ann, line_dash="dash", line_color="#1a7f37",
              annotation_text=f"GARCH equilibrium {eq_vol_ann:.1f}%")
fig.update_layout(title=f"{TICKER}: EWMA vs GARCH conditional volatility (annualised)",
                  yaxis_title="percent per year", height=420)
fig.show()

## Volatility term structure

Only GARCH gives a forward term structure: its forecast average vol by tenor
slopes from today's level toward the equilibrium. EWMA is flat (no mean
reversion). The range estimators are realized, so their "term structure" is a
backward view over increasing trailing windows.

In [ ]:
# Forward term structure for TICKER: GARCH forecast vs the flat EWMA baseline.
# Compare this curve with the implied volatility term structure.
H = 252
var_path = res.forecast(horizon=H, reindex=False).variance.values[-1]   # (pct)^2 per step
ts_vol = np.sqrt(np.cumsum(var_path) / np.arange(1, H + 1) * 252)        # ann %, by tenor
ewma_now = float(pl.Series(ret ** 2).ewm_mean(alpha=1 - LAMBDA, adjust=False).last() ** 0.5
                 * np.sqrt(252))

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(1, H + 1), y=ts_vol, mode="lines",
                         name="GARCH forecast", line=dict(color="#cf222e", width=2)))
fig.add_hline(y=eq_vol_ann, line_dash="dash", line_color="#1a7f37",
              annotation_text=f"equilibrium {eq_vol_ann:.1f}%")
fig.add_hline(y=ewma_now, line_dash="dot", line_color="#1f6feb",
              annotation_text=f"EWMA {ewma_now:.1f}% (flat)")
fig.update_layout(title=f"{TICKER}: volatility term structure (forward)",
                  xaxis_title="horizon (sessions)", yaxis_title="annualised vol %",
                  height=420)
fig.show()

In [ ]:
# Realized (backward) term structure: Yang-Zhang over increasing trailing windows.
d1 = daily.filter(pl.col("ticker") == TICKER)
rows = []
for W in (10, 21, 42, 63, 126, 252):
    w = d1.tail(W)
    kk = 0.34 / (1.34 + (W + 1) / (W - 1))
    yz = w["onr"].var() + kk * w["co"].var() + (1 - kk) * w["rs"].mean()
    rows.append((W, float(np.sqrt(max(yz, 0.0) * 252) * 100)))
tsr = pl.DataFrame(rows, schema=["window", "yz_vol"], orient="row")
fig = px.line(tsr.to_pandas(), x="window", y="yz_vol", markers=True,
              title=f"{TICKER}: realized Yang-Zhang vol by trailing window")
fig.update_layout(xaxis_title="window (sessions)", yaxis_title="annualised vol %", height=380)
fig.show()

### Comparing the estimators

All six current-vol measures side by side, for one ticker and across the liquid set.

In [ ]:
# One ticker: every estimate as a bar. garch_equilibrium is the long-run anchor.
methods = ["garch_now", "ewma_vol_ann", "parkinson_vol", "garman_klass_vol",
           "rogers_satchell_vol", "yang_zhang_vol", "garch_equilibrium"]
row = vol.filter(pl.col("ticker") == TICKER)
fig = px.bar(x=methods, y=[float(row[m][0]) for m in methods],
             title=f"{TICKER}: volatility estimates (annualised %)")
fig.update_layout(xaxis_title="", yaxis_title="annualised vol %", height=400)
fig.show()

In [ ]:
# Cross-section: the spread of each estimator. YZ includes the overnight gap, so it
# usually runs higher than the intraday-only estimators.
methods = ["garch_now", "ewma_vol_ann", "parkinson_vol", "garman_klass_vol",
           "rogers_satchell_vol", "yang_zhang_vol"]
liq = vol.filter(pl.col("converged") & (pl.col("adv_20d_dollars") >= ADV_FLOOR))
long = liq.select(methods).unpivot(variable_name="method", value_name="vol").drop_nulls()
fig = px.box(long.to_pandas(), x="method", y="vol",
             title=f"Volatility estimates across {len(liq)} liquid names")
fig.update_yaxes(range=[0, 150], title="annualised vol %")
fig.update_xaxes(title="")
fig.update_layout(height=440)
fig.show()